In [72]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from typing import List
from tqdm import tqdm
import random
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from sklearn.preprocessing import LabelEncoder
import scipy.sparse as sp

In [73]:
df_users = pd.read_csv('data_original/data_original/users.csv')
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_items = pd.read_csv('data_original/data_original/items.csv')

In [74]:
df_interactions.drop(df_interactions.tail(2).index, inplace=True)

In [75]:
# Конвектируем время в datetime64
df_interactions = pd.read_csv('data_original/data_original/interactions.csv')
df_interactions['last_watch_dt'] = pd.to_datetime(df_interactions['last_watch_dt'], errors='coerce')

In [76]:
# Добавляем новый столбец completed (0<60 , 1>60 просмотрено )
df_interactions['completed'] = (df_interactions['watched_pct'] >= 60).astype(int)


In [77]:
# df_interactions.dropna(subset=['user_id','item_id', 'last_watch_dt', 'watched_pct', 'total_dur','completed'])
# df_users.dropna(subset=['user_id','age', 'income', 'sex', 'kids_flg'])
# df_items.dropna(subset=['item_id', 'content_type', 'title', 'title_orig', 'release_year', 'genres', 'countries', 'for_kids', 'age_rating', 'studios', 'directors', 'actors', 'description', 'keywords'])

In [78]:
print(df_interactions.isna().sum())

user_id            0
item_id            0
last_watch_dt      0
total_dur          0
watched_pct      828
completed          0
dtype: int64


In [79]:
df_interactions = df_interactions[np.isfinite(df_interactions["watched_pct"])]

In [80]:
df_interactions = df_interactions.dropna(subset=['user_id','item_id', 'last_watch_dt', 'watched_pct', 'total_dur','completed'])
# df_users=df_users.dropna(subset=['user_id','age', 'income', 'sex', 'kids_flg'])
# df_items=df_items.dropna(subset=['item_id', 'content_type', 'title', 'title_orig', 'release_year', 'genres', 'countries', 'for_kids', 'age_rating', 'studios', 'directors', 'actors', 'description', 'keywords'])

In [81]:
# df_interactions = df_interactions.dropna(subset=["user_id", "item_id", "watched_pct", "completed"], how="any")

In [82]:
print(df_interactions.isna().sum())
print(df_users.isna().sum())
print(df_items.isna().sum())

user_id          0
item_id          0
last_watch_dt    0
total_dur        0
watched_pct      0
completed        0
dtype: int64
user_id         0
age         14095
income      14776
sex         13831
kids_flg        0
dtype: int64
item_id             0
content_type        0
title               0
title_orig       4745
release_year       98
genres              0
countries          37
for_kids        15397
age_rating          2
studios         14898
directors        1509
actors           2619
description         2
keywords          423
dtype: int64


In [83]:
df_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15963 entries, 0 to 15962
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   item_id       15963 non-null  int64  
 1   content_type  15963 non-null  object 
 2   title         15963 non-null  object 
 3   title_orig    11218 non-null  object 
 4   release_year  15865 non-null  float64
 5   genres        15963 non-null  object 
 6   countries     15926 non-null  object 
 7   for_kids      566 non-null    float64
 8   age_rating    15961 non-null  float64
 9   studios       1065 non-null   object 
 10  directors     14454 non-null  object 
 11  actors        13344 non-null  object 
 12  description   15961 non-null  object 
 13  keywords      15540 non-null  object 
dtypes: float64(3), int64(1), object(10)
memory usage: 1.7+ MB


In [84]:
df_users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840197 entries, 0 to 840196
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   user_id   840197 non-null  int64 
 1   age       826102 non-null  object
 2   income    825421 non-null  object
 3   sex       826366 non-null  object
 4   kids_flg  840197 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 32.1+ MB


In [85]:
df_interactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5475423 entries, 0 to 5476250
Data columns (total 6 columns):
 #   Column         Dtype         
---  ------         -----         
 0   user_id        int64         
 1   item_id        int64         
 2   last_watch_dt  datetime64[ns]
 3   total_dur      int64         
 4   watched_pct    float64       
 5   completed      int32         
dtypes: datetime64[ns](1), float64(1), int32(1), int64(3)
memory usage: 271.5 MB


In [86]:
last_interactions = df_interactions.tail(10)

In [87]:
last_interactions

,user_id,item_id,last_watch_dt,total_dur,watched_pct,completed
5476241,1073802,9927,2021-08-07,6425,97.0,1
5476242,268216,3071,2021-04-21,5752,98.0,1
5476243,497899,9629,2021-05-29,45,1.0,0
5476244,438585,7829,2021-08-02,6804,100.0,1
5476245,786732,4880,2021-05-12,753,0.0,0
5476246,648596,12225,2021-08-13,76,0.0,0
5476247,546862,9673,2021-04-13,2308,49.0,0
5476248,697262,15297,2021-08-20,18307,63.0,1
5476249,384202,16197,2021-04-19,6203,100.0,1
5476250,319709,4436,2021-08-15,3921,45.0,0


In [88]:
user_ids = last_interactions['user_id']
item_ids = last_interactions['item_id']
complete = last_interactions['completed']

In [89]:
from tqdm import tqdm
import time
import numpy as np

def track_progress_tqdm(iterable, desc=None):
    """
    Обертка для отслеживания прогресса с помощью tqdm
    
    Parameters:
    iterable: итерируемый объект
    desc: описание прогресс-бара
    
    Returns:
    Обернутый итератор с прогресс-баром
    """
    return tqdm(iterable, desc=desc, ncols=100, ascii=True)

# Пример использования
def process_data(data):
    results = []
    for item in track_progress_tqdm(data, desc="Обработка данных"):
        # Имитация работы
        time.sleep(0.1)
        results.append(item * 2)
    return results

# Использование
data = list(range(50))
processed = process_data(data)

Обработка данных: 100%|#############################################| 50/50 [00:05<00:00,  9.15it/s]


In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd

def get_user_item_features(user_interactions_train, df_items, df_users):
    """
    Собирает характеристики пользователя на основе его просмотренных фильмов.
    """
    user_items = user_interactions_train[itemIdColumnName].unique()
    if len(user_items) == 0:
        return None, None

    # Получаем данные о фильмах пользователя
    user_items_df = df_items[df_items[itemIdColumnName].isin(user_items)]
    
    # Усредненные характеристики фильмов пользователя
    features = {
        'release_year': user_items_df[releaseYearCol].mean() if not user_items_df[releaseYearCol].isnull().all() else None,
        'genres': set(),
        'actors': set(),
        'directors': set()
    }
    
    # Собираем уникальные жанры, актеров, режиссеров
    for col in [genresCol, actorsCol, directorsCol]:
        if col == genresCol:
            features['genres'] = set([genre for sublist in user_items_df[col].dropna().str.split(', ') for genre in sublist])
        elif col == actorsCol:
            features['actors'] = set([actor for sublist in user_items_df[col].dropna().str.split(', ') for actor in sublist])
        elif col == directorsCol:
            features['directors'] = set([director for sublist in user_items_df[col].dropna().str.split(', ') for director in sublist])
    
    return features, user_items

def calculate_item_similarity(item_id, features, df_items):
    """
    Рассчитывает схожесть фильма с предпочтениями пользователя.
    """
    item_data = df_items[df_items[itemIdColumnName] == item_id]
    if item_data.empty:
        return 0
    
    similarity = 0
    
    # Проверяем год выпуска
    if features['release_year'] is not None and not item_data[releaseYearCol].isnull().values[0]:
        if abs(item_data[releaseYearCol].values[0] - features['release_year']) <= 5:  # Допустимый разброс в 5 лет
            similarity += 0.25
    
    # Проверяем жанры
    item_genres = set(item_data[genresCol].values[0].split(', ')) if not item_data[genresCol].isnull().values[0] else set()
    if features['genres'] and item_genres:
        if len(features['genres'] & item_genres) > 0:
            similarity += 0.25
    
    # Проверяем актеров
    item_actors = set(item_data[actorsCol].values[0].split(', ')) if not item_data[actorsCol].isnull().values[0] else set()
    if features['actors'] and item_actors:
        if len(features['actors'] & item_actors) > 0:
            similarity += 0.25
    
    # Проверяем режиссеров
    item_directors = set(item_data[directorsCol].values[0].split(', ')) if not item_data[directorsCol].isnull().values[0] else set()
    if features['directors'] and item_directors:
        if len(features['directors'] & item_directors) > 0:
            similarity += 0.25
    
    return similarity

def recommend_items_for_user(user_id, df_interactions_train, df_interactions_test, df_items, df_users, k=10, list_all_popular=None):
    """
    Рекомендует фильмы для одного пользователя.
    """
    # Получаем фильмы пользователя в тренировочном наборе
    user_train_items = df_interactions_train[df_interactions_train[userColumnName] == user_id][itemIdColumnName].unique()
    
    # Холодный старт: пользователь отсутствует в тренировочном наборе
    if len(user_train_items) == 0:
        if list_all_popular is not None:
            return list_all_popular[:k]
        return []
    
    # Собираем характеристики пользователя на основе его просмотренных фильмов
    features, _ = get_user_item_features(
        df_interactions_train[df_interactions_train[userColumnName] == user_id],
        df_items,
        df_users
    )
    
    if features is None:
        if list_all_popular is not None:
            return list_all_popular[:k]
        return []
    
    # Находим всех пользователей, которые смотрели хотя бы один из фильмов текущего пользователя
    relevant_users = set()
    for item in user_train_items:
        users_for_item = df_interactions_train[df_interactions_train[itemIdColumnName] == item][userColumnName].unique()
        relevant_users.update(users_for_item)
    
    # Убираем текущего пользователя
    relevant_users.discard(user_id)
    
    if len(relevant_users) == 0:
        if list_all_popular is not None:
            return list_all_popular[:k]
        return []
    
    # Получаем все фильмы, которые смотрели эти пользователи
    relevant_interactions = df_interactions_train[df_interactions_train[userColumnName].isin(relevant_users)]
    candidate_items = set(relevant_interactions[itemIdColumnName].unique())
    
    # Убираем фильмы, которые пользователь уже смотрел
    candidate_items -= set(user_train_items)
    
    # Рассчитываем схожесть для каждого кандидата
    item_scores = {}
    for item in candidate_items:
        score = calculate_item_similarity(item, features, df_items)
        if score > 0:
            item_scores[item] = score
    
    # Сортируем по убыванию score и возвращаем топ-k
    recommended = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)[:k]
    recommended_items = [item for item, _ in recommended]
    
    # Если набралось меньше k, дополняем популярными
    if len(recommended_items) < k and list_all_popular is not None:
        # Берем популярные фильмы, которые не в recommended и не смотрел пользователь
        popular_not_seen = [p for p in list_all_popular if p not in set(recommended_items) and p not in user_train_items]
        recommended_items.extend(popular_not_seen[:k - len(recommended_items)])
    
    return recommended_items

def calculate_mapk_for_users(user_list, df_interactions_train, df_interactions_test, df_items, df_users, k=10, interrupt=False):
    """
    Вычисляет MAP@k для списка пользователей с использованием tqdm для отображения прогресса.
    """
    print("Подготовка данных...")
    
    # Предварительно вычисляем популярные фильмы один раз
    list_all_popular = getPopularItems(df_interactions_train)
    print(f"Популярных фильмов: {len(list_all_popular)}")
    
    # Группируем тестовые данные для быстрого доступа
    print("Группировка тестовых данных...")
    user_test_dict = {}
    for user_id in tqdm(user_list, desc="Группировка тестовых данных", unit="пользователь"):
        user_test_items = df_interactions_test[df_interactions_test[userColumnName] == user_id][itemIdColumnName].unique()
        user_test_dict[user_id] = user_test_items
    
    # Группируем тренировочные данные для быстрого доступа к фильмам пользователя
    print("Группировка тренировочных данных...")
    user_train_items_dict = {}
    for user_id in tqdm(user_list, desc="Группировка тренировочных данных", unit="пользователь"):
        user_train_items = df_interactions_train[df_interactions_train[userColumnName] == user_id][itemIdColumnName].unique()
        if len(user_train_items) > 0:
            user_train_items_dict[user_id] = user_train_items
    
    total_ap = 0
    processed_count = 0
    skipped_count = 0
    
    print("\nНачало расчета MAP@k...")
    
    # Используем tqdm для отображения прогресса
    pbar = tqdm(user_list, desc="Обработка пользователей", unit="пользователь")
    
    for user_id in pbar:
        # Проверяем, есть ли у пользователя достаточно тестовых взаимодействий (>= 2 для MAP@k)
        test_items = user_test_dict.get(user_id, [])
        if len(test_items) < 2:
            skipped_count += 1
            continue
        
        processed_count += 1
        
        # Обновляем описание прогресс-бара
        pbar.set_description(f"Обработка [{processed_count}/{len(user_list)}] | MAP@k: {total_ap/processed_count if processed_count > 0 else 0:.6f}")
        
        # Получаем рекомендации для пользователя
        recommended = recommend_items_for_user(
            user_id, 
            df_interactions_train, 
            df_interactions_test, 
            df_items, 
            df_users, 
            k,
            list_all_popular
        )
        
        if len(recommended) == k:
            # Рассчитываем MAP@k
            ap = ap_k(recommended, test_items, k)
            total_ap += ap
        
        # Прерывание для тестирования
        if interrupt and processed_count >= 100:
            pbar.close()
            print(f"\nПрервано после {processed_count} пользователей")
            break
    
    pbar.close()
    
    # Итоговая статистика
    current_mapk = total_ap / processed_count if processed_count > 0 else 0
    print(f"\n{'='*50}")
    print(f"Статистика:")
    print(f"  Всего пользователей: {len(user_list)}")
    print(f"  Обработано пользователей: {processed_count}")
    print(f"  Пропущено (мало тестовых данных): {skipped_count}")
    print(f"  MAP@{k}: {current_mapk:.6f}")
    print(f"{'='*50}")
    
    return current_mapk

# Функция ap_k (оптимизированная версия)
def ap_k(recommended_list, bought_list, k=5):
    """
    Вычисляет Average Precision at k.
    """
    if len(bought_list) == 0:
        return 0
    
    recommended_list = np.array(recommended_list[:k])
    bought_list = np.array(bought_list)
    
    flags = np.isin(recommended_list, bought_list)
    if not np.any(flags):
        return 0
    
    # Считаем precision для каждой позиции, где есть совпадение
    precisions = []
    correct_count = 0
    for i in range(k):
        if flags[i]:
            correct_count += 1
            precisions.append(correct_count / (i + 1))
    
    return np.mean(precisions) if precisions else 0

# Функция для получения популярных фильмов
def getPopularItems(df):
    """
    Получение популярных items по количеству просмотров.
    """
    items_popularity = df.groupby(itemIdColumnName).size().reset_index(name='count')
    items_popularity = items_popularity.sort_values(by='count', ascending=False)
    return items_popularity[itemIdColumnName].tolist()

# Пример использования:
if __name__ == "__main__":
    # Получаем список всех пользователей из тестового набора
    all_test_users = df_interactions_test[userColumnName].unique()
    print(f"Всего пользователей в тестовом наборе: {len(all_test_users)}")
    
    # Запускаем расчет MAP@k с tqdm
    mapk_value = calculate_mapk_for_users(
        all_test_users, 
        df_interactions_train, 
        df_interactions_test, 
        df_items, 
        df_filtered_users, 
        k=10,
        interrupt=False  # Установите True для тестирования на 100 пользователях
    )
    
    print(f"\nИтоговый MAP@10: {mapk_value:.6f}")

In [ ]:
def precision(recommended_list, bought_list):
    recommended = np.array(recommended_list)
    bought = np.array(bought_list)

    # флаги: какие рекомендованные товары действительно куплены
    flags = np.isin(recommended, bought)

    return flags.sum() / len(recommended)


def precision_at_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    flags = np.isin(recommended, bought)

    return flags.sum() / k


def ap_k(recommended_list, bought_list, k=5):
    recommended = np.array(recommended_list[:k])
    bought = np.array(bought_list)

    # релевантность: рекомендованный товар куплен или нет
    flags = np.isin(recommended, bought)

    # если нет ни одного релевантного — AP = 0
    if flags.sum() == 0:
        return 0.0

    sum_precision = 0.0

    for i in range(k):
        if flags[i]:
            # precision@i+1 (на префиксе)
            precision_i = np.isin(recommended[:i+1], bought).sum() / (i + 1)
            sum_precision += precision_i

    # нормируем на число релевантных объектов в топ-k
    return sum_precision / flags.sum()


def map_k(recommended_list, bought_list, k=5, u=1):
    
    if u == 1:
        return ap_k(recommended_list[u-1], bought_list[u-1], k=5)
    
    sum = 0
    for i in range(0, u):
        ap_k_map = ap_k(recommended_list[i], bought_list[i], k=5)
        sum += ap_k_map

    result = sum / u
    
    return result

In [ ]:
def create_matrix(df, user_col='user_id', item_col='item_id', 
                           value_col='watched_pct'):

    unique_users = sorted(set(df[user_col]))
    unique_items = sorted(set(df[item_col]))

    user_to_index = {user: idx for idx, user in enumerate(unique_users)}
    item_to_index = {item: idx for idx, item in enumerate(unique_items)}

    index_to_user = {idx: user for user, idx in user_to_index.items()}
    index_to_item = {idx: item for item, idx in item_to_index.items()}

    n_users = len(unique_users)
    n_items = len(unique_items)

    user_indices = [user_to_index[u] for u in df[user_col]]
    item_indices = [item_to_index[i] for i in df[item_col]]

    sparse_matrix = csr_matrix((df[value_col], (user_indices, item_indices)), shape=(n_users, n_items))
    
    matrix_df = pd.DataFrame.sparse.from_spmatrix(
        sparse_matrix,
        index=unique_users,
        columns=unique_items
    )
    
    return matrix_df, sparse_matrix, index_to_user, index_to_item, user_to_index, item_to_index 

train_df, train_matrix, index_to_user_train, index_to_item_train, user_to_index_train, item_to_index_train = create_matrix(
    data_train, user_col='user_id', item_col='item_id', value_col='watched_pct'
)
test_df, test_matrix, index_to_user_test, index_to_item_test, user_to_index_test, item_to_index_test  = create_matrix(
    data_test, user_col='user_id', item_col='item_id', value_col='watched_pct'
)



In [ ]:


print("\n" + "="*50)
als_model = ALS(
        n_factors=25,
        alpha=20,
        regularization=0.01,
        iterations=40
    )
    
# Обучение модели
als_model.fit(train_matrix)
    
# Визуализация процесса обучения
als_model.plot_loss()
    
# Оценка модели
print("\nОценка модели:")
    
# Предсказания на тестовой выборке
test_predictions = []
test_actual = []
    
for u in range(test_matrix.shape[0]):
    for i in range(test_matrix.shape[1]):
        if test_matrix[u, i] > 0:
                pred = als_model.predict(u, i)
                test_predictions.append(pred)
                test_actual.append(test_matrix[u, i])
    
# Вычисляем метрики
mse = mean_squared_error(test_actual, test_predictions)
rmse = np.sqrt(mse)


print(f"RMSE на тестовой выборке: {rmse:.4f}")



In [ ]:
data_test_group = data_test.groupby(data_test['user_id'])['item_id'].unique().reset_index()
data_test_group.columns = ['user_id', 'item_id']
data_test_group

In [ ]:
userids = data_test_group['user_id'].values
 
userids_test = np.arange(len(userids))

In [ ]:


def recommendations__original_ids(user_matrix_index, item_recommendations):
    original_item_ids = [index_to_item_train[idx] for idx in item_recommendations]
    return user_matrix_index, original_item_ids



In [ ]:
recommended = []
for i in range(len(userids_test)):
    recommended_items, scores = als_model.recommend(userids_test[i], n_recommendations=10)
    recommended.append({'item_id_pred': recommended_items})

result = pd.DataFrame(recommended, userids).reset_index()
result

In [ ]:
recom =[]
for i in range(len(result)):
    user, recommend = recommendations__original_ids(userids[i], result['item_id_pred'][i])
    recom.append({'item_id_pred': recommend})

result_pred = pd.DataFrame(recom, userids).reset_index()
result_pred

In [ ]:


print('als_map_k =', map_k(result_pred['item_id_pred'], data_test_group['item_id'], k=10, u=len(data_test_group)))

